### Data Preprocessing

### Importing libraries

In [ ]:
import polars as pl
import numpy as np
import plotly.express as pxi
import plotly.graph_objects as go
import polars as pl
import plotly.express as pxi
import plotly.graph_objects as go
from pred_irr_comp import (
    PROCESSED_DATA_DIR,
    TARGET_COL,
    LABEL_MAP,
    ID_COLS,
    split_columns,
    load_train_test,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import pickle
import os
from skimpy import skim
import shap
import lightgbm as lgb
from skimpy import skim_get_figure
from pred_irr_comp.custom.skim_extra import skim_get_figure_sized
from pred_irr_comp.custom.plot_extra import numeric_kde_figure

In [ ]:
#Train and Test Data
train_df, test_df = load_train_test()

#Splitting columns
num_cols, cat_cols = split_columns(train_df)

In [ ]:
print(f"Numerical columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")
print('Target column: ',TARGET_COL)

In [ ]:
train_df.head()

In [ ]:
# Remove ID columns
train_df = train_df.drop(ID_COLS)
test_df = test_df.drop(ID_COLS)

In [ ]:
# growth_order   = [['Sowing', 'Vegetative', 'Flowering', 'Harvest']]
# mulching_order = [['No', 'Yes']]

# onehot_cols = [
#     'Soil_Type', 'Crop_Type', 'Season',
#     'Irrigation_Type', 'Water_Source', 'Region',
# ]

# preprocess = ColumnTransformer(
#     transformers=[
#         ('num',    StandardScaler(),num_cols),
#         ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False),onehot_cols),
#         ('growth', OrdinalEncoder(categories=growth_order),['Crop_Growth_Stage']),
#         ('mulch',  OrdinalEncoder(categories=mulching_order),['Mulching_Used']),
#     ],
#     remainder='drop',
# )

# y_train = train_df[TARGET_COL].replace_strict(LABEL_MAP).to_numpy()
# X_train = preprocess.fit_transform(train_df.drop(TARGET_COL).to_pandas())
# X_test  = preprocess.transform(test_df.to_pandas())

## Preprocessed Data - 1.1

- No Feature Engineering, just simple conversion of categorical features to numerical features via encoding
- Save this preprocessed data in directory


In [ ]:
# directory = PROCESSED_DATA_DIR / 'preprocessed_v1.1'
# directory.mkdir(parents=True, exist_ok=True)

# with open(directory / 'X_train.pkl', 'wb') as f:
#     pickle.dump(X_train, f)
# with open(directory / 'X_test.pkl', 'wb') as f:
#     pickle.dump(X_test, f)
# with open(directory / 'y_train.pkl', 'wb') as f:
#     pickle.dump(y_train, f)

## Preprocessed Data - 1.2

- Feature Engineering on the data(golden features)

In [ ]:
# Original Dataframe
train_df

### Creating the Golden features

In [ ]:
KC_TABLE = {
    ('Rice',      'Sowing'):    1.05, ('Rice',      'Vegetative'): 1.10,
    ('Rice',      'Flowering'): 1.20, ('Rice',      'Harvest'):    0.90,
    ('Sugarcane', 'Sowing'):    0.40, ('Sugarcane', 'Vegetative'): 0.85,
    ('Sugarcane', 'Flowering'): 1.25, ('Sugarcane', 'Harvest'):    0.75,
    ('Wheat',     'Sowing'):    0.30, ('Wheat',     'Vegetative'): 0.70,
    ('Wheat',     'Flowering'): 1.15, ('Wheat',     'Harvest'):    0.40,
    ('Maize',     'Sowing'):    0.30, ('Maize',     'Vegetative'): 0.70,
    ('Maize',     'Flowering'): 1.20, ('Maize',     'Harvest'):    0.60,
    ('Potato',    'Sowing'):    0.50, ('Potato',    'Vegetative'): 0.80,
    ('Potato',    'Flowering'): 1.15, ('Potato',    'Harvest'):    0.75,
    ('Cotton',    'Sowing'):    0.35, ('Cotton',    'Vegetative'): 0.75,
    ('Cotton',    'Flowering'): 1.15, ('Cotton',    'Harvest'):    0.50,
}
KC_MAP = {f'{c}|{s}': v for (c, s), v in KC_TABLE.items()}

SOIL_AWC   = {'Sandy': 70.0,  'Loamy': 150.0, 'Silt': 200.0, 'Clay': 180.0}
SOIL_DRAIN = {'Sandy': 0.9,   'Loamy': 0.6,   'Silt': 0.4,   'Clay': 0.2}

CROP_WATER_DEMAND = {
    'Rice': 3.0, 'Sugarcane': 2.5,
    'Maize': 2.0, 'Wheat': 1.8, 'Cotton': 1.8,
    'Potato': 1.5,
}

IRR_EFFICIENCY = {'Drip': 0.90, 'Sprinkler': 0.75, 'Canal': 0.55, 'Rainfed': 0.30}
SOURCE_RELIABILITY = {'River': 3.0, 'Reservoir': 2.5, 'Groundwater': 2.0, 'Rainwater': 0.5}


def engineer_features(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .with_columns(
            SVP=(0.6108 * (17.27 * pl.col('Temperature_C')
                           / (pl.col('Temperature_C') + 237.3)).exp()),
        )
        .with_columns(
            VPD=pl.col('SVP') * (1 - pl.col('Humidity') / 100),
            AVP=pl.col('SVP') * pl.col('Humidity') / 100,
            ETo_proxy=(
                0.0023
                * (pl.col('Temperature_C') + 17.8)
                * (pl.col('Sunlight_Hours') * (1 - pl.col('Humidity') / 100) + 1e-6).sqrt()
                * (1 + 0.1 * pl.col('Wind_Speed_kmh') / 10)
            ),
        )
        .with_columns(
            Kc=pl.concat_str([pl.col('Crop_Type'), pl.lit('|'), pl.col('Crop_Growth_Stage')])
                .replace_strict(KC_MAP, default=0.80),
        )
        .with_columns(
            ETc=pl.col('Kc') * pl.col('ETo_proxy'),
        )
        .with_columns(
            ETc_effective=pl.col('ETc')
                          * (1 - 0.25 * (pl.col('Mulching_Used') == 'Yes').cast(pl.Float64)),
        )
        .with_columns(
            Effective_Rainfall_mm=pl.when(pl.col('Rainfall_mm') < 250)
                .then(pl.col('Rainfall_mm') * (125 - 0.2 * pl.col('Rainfall_mm')) / 125)
                .otherwise(125 + 0.1 * pl.col('Rainfall_mm')),
            Soil_AWC=pl.col('Soil_Type').replace_strict(SOIL_AWC, default=130.0),
            Soil_Drainage=pl.col('Soil_Type').replace_strict(SOIL_DRAIN, default=0.5),
            Crop_Water_Demand=pl.col('Crop_Type').replace_strict(CROP_WATER_DEMAND, default=2.0),
            Irrigation_Efficiency=pl.col('Irrigation_Type').replace_strict(IRR_EFFICIENCY, default=0.5),
            Source_Reliability=pl.col('Water_Source').replace_strict(SOURCE_RELIABILITY, default=1.5),
        )
        .with_columns(
            Water_Balance=pl.col('Effective_Rainfall_mm') + pl.col('Previous_Irrigation_mm')
                          - pl.col('ETc_effective') * 30,
            Aridity_Index=pl.col('Effective_Rainfall_mm')
                          / (pl.col('ETc_effective') * 30 + 1e-3),
            Soil_Moisture_Deficit=pl.col('Soil_AWC') - pl.col('Soil_Moisture') * 10,
            Relative_Moisture=pl.col('Soil_Moisture') * 10 / (pl.col('Soil_AWC') + 1e-3),
            Heat_Stress=pl.max_horizontal(pl.col('Temperature_C') - 30, pl.lit(0.0)),
            Cold_Stress=pl.max_horizontal(pl.lit(15.0) - pl.col('Temperature_C'), pl.lit(0.0)),
            Extreme_Heat=pl.max_horizontal(pl.col('Temperature_C') - 35, pl.lit(0.0)) ** 2,
            Salinity_Stress=pl.max_horizontal(pl.col('Electrical_Conductivity') - 2.0, pl.lit(0.0)),
            pH_Stress=(pl.col('Soil_pH') - 6.5).abs(),
            Acid_Stress=pl.max_horizontal(pl.lit(5.5) - pl.col('Soil_pH'), pl.lit(0.0)),
            Alkaline_Stress=pl.max_horizontal(pl.col('Soil_pH') - 7.5, pl.lit(0.0)),
        )
        .with_columns(
            Rainfall_Deficit=pl.max_horizontal(
                pl.col('ETc_effective') * 30 - pl.col('Effective_Rainfall_mm'),
                pl.lit(0.0),
            ),
            Rainfall_Excess=pl.max_horizontal(
                pl.col('Effective_Rainfall_mm') - pl.col('ETc_effective') * 30,
                pl.lit(0.0),
            ),
        )
        .with_columns(
            Wind_x_VPD=pl.col('Wind_Speed_kmh') * pl.col('VPD'),
            Sun_x_Temp=pl.col('Sunlight_Hours') * pl.col('Temperature_C'),
            Total_Water_Need_m3=pl.col('ETc_effective')
                                * pl.col('Field_Area_hectare') * 10,
            Net_Irrigation_Need=pl.max_horizontal(
                pl.col('ETc_effective') * 30
                - pl.col('Effective_Rainfall_mm')
                - pl.col('Previous_Irrigation_mm') * pl.col('Irrigation_Efficiency'),
                pl.lit(0.0),
            ),
            Moisture_x_Carbon=pl.col('Soil_Moisture') * pl.col('Organic_Carbon'),
            Rain_x_Drainage=pl.col('Effective_Rainfall_mm') * pl.col('Soil_Drainage'),
            AWC_x_OC=pl.col('Soil_AWC') * pl.col('Organic_Carbon'),
            Efficient_Irrigation=pl.col('Irrigation_Type')
                                   .is_in(['Drip', 'Sprinkler']).cast(pl.Float64),
            Is_Monsoon=(pl.col('Season') == 'Kharif').cast(pl.Float64),
            Is_Dry_Season=pl.col('Season').is_in(['Rabi', 'Zaid']).cast(pl.Float64),
            Prev_Irr_Effective=pl.col('Previous_Irrigation_mm') * pl.col('Irrigation_Efficiency'),
            Demand_vs_Source=pl.col('Crop_Water_Demand') / (pl.col('Source_Reliability') + 1e-3),
            Log_Rainfall=pl.col('Rainfall_mm').log1p(),
            Log_Field_Area=pl.col('Field_Area_hectare').log1p(),
            Log_Prev_Irrigation=pl.col('Previous_Irrigation_mm').log1p(),
        )
    )

### Apply feature engineering to train and test

In [ ]:
train_df_fe = engineer_features(train_df)
test_df_fe  = engineer_features(test_df)

NEW_NUM_FEATURES = [
    'SVP', 'VPD', 'AVP', 'ETo_proxy',
    'Kc', 'ETc', 'ETc_effective',
    'Effective_Rainfall_mm', 'Soil_AWC', 'Soil_Drainage',
    'Crop_Water_Demand', 'Irrigation_Efficiency', 'Source_Reliability',
    'Water_Balance', 'Aridity_Index',
    'Soil_Moisture_Deficit', 'Relative_Moisture',
    'Heat_Stress', 'Cold_Stress', 'Extreme_Heat',
    'Salinity_Stress', 'pH_Stress', 'Acid_Stress', 'Alkaline_Stress',
    'Rainfall_Deficit', 'Rainfall_Excess',
    'Wind_x_VPD', 'Sun_x_Temp', 'Total_Water_Need_m3', 'Net_Irrigation_Need',
    'Moisture_x_Carbon', 'Rain_x_Drainage', 'AWC_x_OC',
    'Efficient_Irrigation', 'Is_Monsoon', 'Is_Dry_Season',
    'Prev_Irr_Effective', 'Demand_vs_Source',
    'Log_Rainfall', 'Log_Field_Area', 'Log_Prev_Irrigation',
]

print(f'train shape after FE: {train_df_fe.shape}')
print(f'test  shape after FE: {test_df_fe.shape}')
print(f'new numerical features added: {len(NEW_NUM_FEATURES)}')

In [ ]:
skim(train_df_fe)

In [ ]:
train_df_fe

### Encode v1.2 and save

Same encoding scheme as v1.1 (OHE for nominal, ordinal for growth stage / mulching) — the only difference is the enlarged numeric block from `engineer_features`. Originals are kept so tree models can choose.

In [ ]:
growth_order   = [['Sowing', 'Vegetative', 'Flowering', 'Harvest']]
mulching_order = [['No', 'Yes']]

onehot_cols = [
    'Soil_Type', 'Crop_Type', 'Season',
    'Irrigation_Type', 'Water_Source', 'Region',
]
num_cols_v12 = num_cols + NEW_NUM_FEATURES

preprocess_v12 = ColumnTransformer(
    transformers=[
        # ('num',    StandardScaler(), num_cols_v12), # standrd Scaler not used preferred for tree models - wasted computation
        ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False), onehot_cols),
        ('growth', OrdinalEncoder(categories=growth_order),   ['Crop_Growth_Stage']),
        ('mulch',  OrdinalEncoder(categories=mulching_order), ['Mulching_Used']),
    ],
    remainder='drop',
)

y_train_v12 = train_df_fe[TARGET_COL].replace_strict(LABEL_MAP).to_numpy()
X_train_v12 = preprocess_v12.fit_transform(train_df_fe.drop(TARGET_COL).to_pandas())
X_test_v12  = preprocess_v12.transform(test_df_fe.to_pandas())

print(f'X_train_v12: {X_train_v12.shape}')
print(f'X_test_v12 : {X_test_v12.shape}')
print(f'numeric features: {len(num_cols_v12)} | total columns: {X_train_v12.shape[1]}')

In [ ]:
directory_v12 = PROCESSED_DATA_DIR / 'preprocessed_v1.2'
directory_v12.mkdir(parents=True, exist_ok=True)

with open(directory_v12 / 'X_train.pkl', 'wb') as f:
    pickle.dump(X_train_v12, f)
with open(directory_v12 / 'X_test.pkl', 'wb') as f:
    pickle.dump(X_test_v12, f)
with open(directory_v12 / 'y_train.pkl', 'wb') as f:
    pickle.dump(y_train_v12, f)
with open(directory_v12 / 'feature_names.pkl', 'wb') as f:
    pickle.dump(list(preprocess_v12.get_feature_names_out()), f)

print(f'Saved preprocessed_v1.2 -> {directory_v12}')

## Preprocessed Data - 1.3

- Feature Importance Analysis using SHAP Analysis

In [ ]:
X_train_v12 = pickle.load(open(directory_v12 / 'X_train.pkl', 'rb'))
y_train_v12 = pickle.load(open(directory_v12 / 'y_train.pkl', 'rb'))

rng = np.random.default_rng(42)
idx = rng.choice(len(X_train_v12), size=10_000, replace=False)
X_sample = X_train_v12.iloc[idx] if hasattr(X_train_v12, 'iloc') else X_train_v12[idx]
y_sample = y_train_v12[idx]

lgbm = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    subsample=0.9, colsample_bytree=0.9,
    objective='multiclass', num_class=3,
    n_jobs=-1, random_state=42, verbose=-1,
)
lgbm.fit(X_train_v12, y_train_v12)
explainer = shap.TreeExplainer(lgbm)
shap_values = explainer(X_sample)          # shape (n, features, 3)
shap.summary_plot(shap_values, X_sample, plot_type='bar')

In [ ]:
# Names oif the features -->
feature_names = pickle.load(open(directory_v12 / 'feature_names.pkl', 'rb'))
feature_names

### Preprocessed Data - 1.3
- Data is augmented, more carefully selected and added some features.


In [ ]:
import os 
os.makedirs("../data/processed/preprocessed_v1.3", exist_ok=True)

In [ ]:
train_df_fe.head()

In [ ]:
skim_get_figure_sized(
    train_df_fe,
    "../reports/figures/summary_table_v1.2.svg",
    console_width=145,
    format="svg",
)

In [ ]:
fig = numeric_kde_figure(
    train_df_fe,
    ncols=6,
    show_kde=False,
    default_bins=60,
    nbins={},
    drop_cols=(),   # Prevent dropping 'id' since it's not present
)
fig.show()

In [ ]:
KC_TABLE = {
    ('Rice',      'Sowing'):    1.05, ('Rice',      'Vegetative'): 1.10,
    ('Rice',      'Flowering'): 1.20, ('Rice',      'Harvest'):    0.90,
    ('Sugarcane', 'Sowing'):    0.40, ('Sugarcane', 'Vegetative'): 0.85,
    ('Sugarcane', 'Flowering'): 1.25, ('Sugarcane', 'Harvest'):    0.75,
    ('Wheat',     'Sowing'):    0.30, ('Wheat',     'Vegetative'): 0.70,
    ('Wheat',     'Flowering'): 1.15, ('Wheat',     'Harvest'):    0.40,
    ('Maize',     'Sowing'):    0.30, ('Maize',     'Vegetative'): 0.70,
    ('Maize',     'Flowering'): 1.20, ('Maize',     'Harvest'):    0.60,
    ('Potato',    'Sowing'):    0.50, ('Potato',    'Vegetative'): 0.80,
    ('Potato',    'Flowering'): 1.15, ('Potato',    'Harvest'):    0.75,
    ('Cotton',    'Sowing'):    0.35, ('Cotton',    'Vegetative'): 0.75,
    ('Cotton',    'Flowering'): 1.15, ('Cotton',    'Harvest'):    0.50,
}
KC_MAP = {f'{c}|{s}': v for (c, s), v in KC_TABLE.items()}

SOIL_AWC   = {'Sandy': 70.0,  'Loamy': 150.0, 'Silt': 200.0, 'Clay': 180.0}
SOIL_DRAIN = {'Sandy': 0.9,   'Loamy': 0.6,   'Silt': 0.4,   'Clay': 0.2}

CROP_WATER_DEMAND = {
    'Rice': 3.0, 'Sugarcane': 2.5,
    'Maize': 2.0, 'Wheat': 1.8, 'Cotton': 1.8,
    'Potato': 1.5,
}

IRR_EFFICIENCY = {'Drip': 0.90, 'Sprinkler': 0.75, 'Canal': 0.55, 'Rainfed': 0.30}
SOURCE_RELIABILITY = {'River': 3.0, 'Reservoir': 2.5, 'Groundwater': 2.0, 'Rainwater': 0.5}


def engineer_features(df: pl.DataFrame) -> pl.DataFrame:
    # 4 boolean numeric features based on threshold insights
    df = df.with_columns([
        (pl.col('Soil_Moisture') < 25).cast(pl.Int8).alias('soil_lt_25'),
        (pl.col('Temperature_C') > 30).cast(pl.Int8).alias('temp_gt_30'),
        (pl.col('Rainfall_mm') < 300).cast(pl.Int8).alias('rain_lt_300'),
        (pl.col('Wind_Speed_kmh') > 10).cast(pl.Int8).alias('wind_gt_10'),
    ])
    return (
        df
        .with_columns(
            # SVP and AVP DROPPED, keep only VPD
            VPD=(
                0.6108 * (17.27 * pl.col('Temperature_C')
                / (pl.col('Temperature_C') + 237.3)).exp()
                * (1 - pl.col('Humidity') / 100)
            ),
        )
        .with_columns(
            # ETo_proxy and ETc DROPPED, only compute Kc and ETc_effective
            Kc=pl.concat_str([pl.col('Crop_Type'), pl.lit('|'), pl.col('Crop_Growth_Stage')])
                .replace_strict(KC_MAP, default=0.80),
        )
        .with_columns(
            # For ETc_effective, need ETo_proxy calculation locally
            # ETo_proxy intermediate, but do not keep
            _ETo_proxy=(
                0.0023
                * (pl.col('Temperature_C') + 17.8)
                * (pl.col('Sunlight_Hours') * (1 - pl.col('Humidity') / 100) + 1e-6).sqrt()
                * (1 + 0.1 * pl.col('Wind_Speed_kmh') / 10)
            ),
        )
        .with_columns(
            ETc_effective=(pl.col('Kc') * pl.col('_ETo_proxy'))
                          * (1 - 0.25 * (pl.col('Mulching_Used') == 'Yes').cast(pl.Float64)),
        )
        .drop(['_ETo_proxy'])  # Remove the temporary column
        .with_columns(
            Effective_Rainfall_mm=pl.when(pl.col('Rainfall_mm') < 250)
                .then(pl.col('Rainfall_mm') * (125 - 0.2 * pl.col('Rainfall_mm')) / 125)
                .otherwise(125 + 0.1 * pl.col('Rainfall_mm')),
            Soil_AWC=pl.col('Soil_Type').replace_strict(SOIL_AWC, default=130.0),
            # Soil_Drainage DROPPED
            Crop_Water_Demand=pl.col('Crop_Type').replace_strict(CROP_WATER_DEMAND, default=2.0),
            Irrigation_Efficiency=pl.col('Irrigation_Type').replace_strict(IRR_EFFICIENCY, default=0.5),
            Source_Reliability=pl.col('Water_Source').replace_strict(SOURCE_RELIABILITY, default=1.5),
        )
        .with_columns(
            Water_Balance=pl.col('Effective_Rainfall_mm') + pl.col('Previous_Irrigation_mm')
                          - pl.col('ETc_effective') * 30,
            Aridity_Index=pl.col('Effective_Rainfall_mm')
                          / (pl.col('ETc_effective') * 30 + 1e-3),
            Soil_Moisture_Deficit=pl.col('Soil_AWC') - pl.col('Soil_Moisture') * 10,
            Relative_Moisture=pl.col('Soil_Moisture') * 10 / (pl.col('Soil_AWC') + 1e-3),
            # DROPPED:
            # Heat_Stress, Cold_Stress, Extreme_Heat, Acid_Stress, Alkaline_Stress
            Salinity_Stress=pl.max_horizontal(pl.col('Electrical_Conductivity') - 2.0, pl.lit(0.0)),
            pH_Stress=(pl.col('Soil_pH') - 6.5).abs(),
        )
        .with_columns(
            # DROPPED: Rainfall_Deficit
            Rainfall_Excess=pl.max_horizontal(
                pl.col('Effective_Rainfall_mm') - pl.col('ETc_effective') * 30,
                pl.lit(0.0),
            ),
        )
        .with_columns(
            Wind_x_VPD=pl.col('Wind_Speed_kmh') * pl.col('VPD'),
            Sun_x_Temp=pl.col('Sunlight_Hours') * pl.col('Temperature_C'),
            Total_Water_Need_m3=pl.col('ETc_effective')
                                * pl.col('Field_Area_hectare') * 10,
            # DROPPED: Net_Irrigation_Need
            Moisture_x_Carbon=pl.col('Soil_Moisture') * pl.col('Organic_Carbon'),
            Rain_x_Drainage=pl.col('Effective_Rainfall_mm') * pl.col('Soil_Type').replace_strict(SOIL_DRAIN, default=0.5),
            AWC_x_OC=pl.col('Soil_AWC') * pl.col('Organic_Carbon'),
            Efficient_Irrigation=pl.col('Irrigation_Type')
                                   .is_in(['Drip', 'Sprinkler']).cast(pl.Float64),
            # DROPPED: Is_Monsoon
            Is_Dry_Season=pl.col('Season').is_in(['Rabi', 'Zaid']).cast(pl.Float64),
            Prev_Irr_Effective=pl.col('Previous_Irrigation_mm') * pl.col('Irrigation_Efficiency'),
            Demand_vs_Source=pl.col('Crop_Water_Demand') / (pl.col('Source_Reliability') + 1e-3),
            Log_Rainfall=pl.col('Rainfall_mm').log1p(),
            Log_Field_Area=pl.col('Field_Area_hectare').log1p(),
            Log_Prev_Irrigation=pl.col('Previous_Irrigation_mm').log1p(),
        )
    )

In [ ]:
train_df_new_fe = engineer_features(train_df)
test_df_new_fe  = engineer_features(test_df)

# Only include relevant features: remove intermediate, dropped, or engineered features that are not retained in final dataframe.
NEW_NUM_FEATURES = [
    # Atmospheric & crop demand
    'VPD', 'Kc', 'ETc_effective',
    # Supply
    'Effective_Rainfall_mm', 'Soil_AWC',
    # Lookup-from-categorical numerics
    'Crop_Water_Demand', 'Irrigation_Efficiency', 'Source_Reliability',
    # Deficit / balance (the money features)
    'Water_Balance', 'Aridity_Index',
    'Soil_Moisture_Deficit', 'Relative_Moisture',
    # Stress
    'Salinity_Stress', 'pH_Stress',
    # Excess
    'Rainfall_Excess',
    # Interactions
    'Wind_x_VPD', 'Sun_x_Temp', 'Total_Water_Need_m3',
    'Moisture_x_Carbon', 'Rain_x_Drainage', 'AWC_x_OC',
    # Flags
    'Efficient_Irrigation', 'Is_Dry_Season',
    # Effective previous irrigation
    'Prev_Irr_Effective', 'Demand_vs_Source',
    # Threshold flags (you create these in the function — don't forget them)
    'soil_lt_25', 'temp_gt_30', 'rain_lt_300', 'wind_gt_10',
    # Logs (optional — see note below)
    'Log_Rainfall', 'Log_Field_Area', 'Log_Prev_Irrigation',
]

print(f'train shape after FE: {train_df_new_fe.shape}')
print(f'test  shape after FE: {test_df_new_fe.shape}')
print(f'new numerical features added: {len(NEW_NUM_FEATURES)}')

In [ ]:
growth_order   = [['Sowing', 'Vegetative', 'Flowering', 'Harvest']]
mulching_order = [['No', 'Yes']]

onehot_cols = [
    'Soil_Type', 'Crop_Type', 'Season',
    'Irrigation_Type', 'Water_Source', 'Region',
]
num_cols_v13 = num_cols + NEW_NUM_FEATURES

preprocess_v13 = ColumnTransformer(
    transformers=[
        ('num',    'passthrough', num_cols_v13),
        ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False), onehot_cols),
        ('growth', OrdinalEncoder(categories=growth_order),   ['Crop_Growth_Stage']),
        ('mulch',  OrdinalEncoder(categories=mulching_order), ['Mulching_Used']),
    ],
    remainder='drop',
)

y_train_v13 = train_df_new_fe[TARGET_COL].replace_strict(LABEL_MAP).to_numpy()
X_train_v13 = preprocess_v13.fit_transform(train_df_new_fe.drop(TARGET_COL).to_pandas())
X_test_v13  = preprocess_v13.transform(test_df_new_fe.to_pandas())

print(f'X_train_v13: {X_train_v13.shape}')
print(f'X_test_v13 : {X_test_v13.shape}')

# Calculate the actual numeric, one-hot-encoded, ordinal, and total features
n_num_features = len(num_cols_v13)
ohe_feat_count = preprocess_v13.named_transformers_['ohe'].get_feature_names_out(onehot_cols).shape[0]
growth_feat_count = 1  # single ordinal
mulch_feat_count = 1   # single ordinal
total_dim = n_num_features + ohe_feat_count + growth_feat_count + mulch_feat_count

print(f'Numeric (passthrough) features: {n_num_features}')
print(f'One-hot-encoded features: {ohe_feat_count}')
print(f'Growth stage (ordinal): {growth_feat_count}')
print(f'Mulching used (ordinal): {mulch_feat_count}')
print(f'Expected total columns: {total_dim} | Actual: {X_train_v13.shape[1]}')

if n_num_features > X_train_v13.shape[1]:
    print("[WARNING] Number of numeric features > total output columns! You may be dropping columns or mis-aligning your transformer. Double check num_cols_v13 and ColumnTransformer configs.")

In [ ]:
directory_v13 = PROCESSED_DATA_DIR / 'preprocessed_v1.3'
directory_v13.mkdir(parents=True, exist_ok=True)

with open(directory_v13 / 'X_train.pkl', 'wb') as f:
    pickle.dump(X_train_v13, f)
with open(directory_v13 / 'X_test.pkl', 'wb') as f:
    pickle.dump(X_test_v13, f)
with open(directory_v13 / 'y_train.pkl', 'wb') as f:
    pickle.dump(y_train_v13, f)
with open(directory_v13 / 'feature_names.pkl', 'wb') as f:
    pickle.dump(list(preprocess_v13.get_feature_names_out()), f)
print(f'Saved preprocessed_v1.3 -> {directory_v13}')

In [ ]:
# Preprocessed Data - 1.5